# Estimate mesh error before using a result
### Richardson extrapolation and the grid convergence index

Estimate the mesh error of a slab centerline temperature using three systematically refined calculations. Richardson extrapolation assumes a leading error proportional to $h^p$; three meshes let you estimate $p$ instead of assuming it. The grid convergence index (GCI) applies a safety factor to the fine-grid difference, but it does not cover every modeling or input uncertainty. Examine its assumptions, especially for oscillatory convergence or coarse meshes. The smooth manufactured problem here supplies an exact check.

**Today's goal:** estimate the mesh error of the centerline temperature from three resolutions.

**Success criteria:** observed order between 1.9 and 2.1; extrapolated temperature rise within 0.001 K of the exact value; and a correctly labeled fine-grid error estimate. No earlier assignment is required.

**Time:** 40 minutes of practice and 25 minutes of reading. Run cells from top to bottom with NumPy, Matplotlib, and IPython.

**Notation:** $T_N$ is the numerical temperature field on an $N$-interval mesh. $f_N=T_N(L/2)-300\,\mathrm{K}$ is the computed centerline temperature rise; the exact rise is 10 K. Keep signed mesh differences for extrapolation and use absolute values for error magnitudes.

## 1. Governing model and reference solution

**Educational steady slab:**

$$L=0.10\,\mathrm{m},\qquad k=10\,\mathrm{W\,m^{-1}\,K^{-1}},\qquad T(0)=T(L)=300\,\mathrm{K}.$$

There is no advection or radiation. Solve

$$-k\frac{d^2T}{dx^2}=q'''(x),\qquad 0<x<L,$$
$$q'''(x)=k(10\,\mathrm{K})\left(\frac{\pi}{L}\right)^2\sin\left(\frac{\pi x}{L}\right).$$

The manufactured exact solution is

$$T^*(x)=300\,\mathrm{K}+(10\,\mathrm{K})\sin\left(\frac{\pi x}{L}\right).$$

Use centered differences with $N=10,20,40$ equal intervals and NumPy's direct linear solve. Track the centerline temperature rise

$$f_N=T_N(L/2)-300\,\mathrm{K}.$$

**Modeling task:** sketch the equation and derive the dimensional source units and exact outward-flux balance. Explain why each even-interval mesh samples the same center location $x=L/2$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

L = 0.10  # m
k = 10.0  # W/(m K)
T_LEFT = T_RIGHT = 300.0  # K
AMPLITUDE = 10.0  # K
MESHES = (10, 20, 40)  # equal intervals; all even so each includes x=L/2

plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})


def exact_temperature(x):
    """Return the manufactured exact temperature in kelvin."""
    return T_LEFT + AMPLITUDE * np.sin(np.pi * x / L)


def source_term(x):
    """Return the volumetric heat-generation rate in W/m^3."""
    return k * AMPLITUDE * (np.pi / L)**2 * np.sin(np.pi * x / L)

## 2. Discretization and conditions

Use $N$ equal intervals, $N+1$ nodes, and $h=L/N$. At interior node $i$, centered differences give

$$-k\frac{T_{i-1}-2T_i+T_{i+1}}{h^2}=q'''(x_i).$$

Only the $N-1$ interior temperatures are unknown. The two endpoint temperatures are prescribed, so their stencil contributions move to the right-hand side of the interior system.

**Derive in your equation sketch:** the diagonal and adjacent entries of $A$, the source entries in $b$, and the first/last right-hand-side contributions from $T(0)$ and $T(L)$. Keep the $h^{-2}$ factor in the matrix consistent with the source scaling.

**Check:** verify the source units are $\mathrm{W\,m^{-3}}$ and show that the exact total outward heat flux equals $\int_0^Lq'''(x)\,dx$ per unit area.

In [2]:
def solve(N):
    """Return nodal coordinates x and temperature T for N equal intervals."""
    h = L / N
    x = np.linspace(0.0, L, N + 1)
    interior_count = N - 1
    A = np.zeros((interior_count, interior_count))
    b = source_term(x[1:-1]).copy()
    A = (2.0 * np.eye(interior_count)
         - np.eye(interior_count, k=1)
         - np.eye(interior_count, k=-1)) / h**2
    # TODO: Add the known left/right endpoint contributions to b.
    T = np.full(N + 1, np.nan)
    T[0], T[-1] = T_LEFT, T_RIGHT
    # TODO: Solve A @ T[1:-1] = b with np.linalg.solve.
    raise NotImplementedError("Complete the stencil, boundary contributions, and solve.")

## 3. Richardson extrapolation and error estimate

Assume the centerline result has a leading mesh error proportional to $h^p$. The refinement ratio is $r=2$. Use the three computed rises $f_{10},f_{20},f_{40}$ to estimate the observed order

$$p=\log_2\left|\frac{f_{10}-f_{20}}{f_{20}-f_{40}}\right|.$$

Preserve the sign of the fine-grid difference in the extrapolation:

$$f_{\mathrm{ext}}=f_{40}+\frac{f_{40}-f_{20}}{2^p-1}.$$

The requested fine-grid GCI estimate is

$$G=\frac{1.25|f_{40}-f_{20}|}{2^p-1},\qquad \frac{G}{|f_{40}|}\text{ is a fraction.}$$

Report $f_{\mathrm{ext}}$ and $G$ in kelvin, label $G$ as an estimated fine-grid error, and compare it with the exact fine-grid error $|f_{40}-10\,\mathrm{K}|$. Treat $G$ as a numerical estimate, not a guaranteed confidence interval.

**Your task:** calculate the three quantities from the results below. Keep signed differences for $f_{\mathrm{ext}}$; use absolute values only where shown.

In [3]:
mesh_results = {}
for N in MESHES:
    x, T = solve(N)
    center_index = N // 2
    assert np.isclose(x[center_index], L / 2), "Mesh must include the slab center."
    f = T[center_index] - T_LEFT
    mesh_results[N] = {"h": L / N, "x": x, "T": T, "f": f}

f10, f20, f40 = (mesh_results[N]["f"] for N in MESHES)

# TODO: Estimate p from the three centerline rises.
# TODO: Compute f_ext using the signed difference (f40 - f20).
# TODO: Compute G and G / abs(f40), then compare G with abs(f40 - AMPLITUDE).

NotImplementedError: Complete the stencil, boundary contributions, and solve.

## 4. Results and convergence

Compare the centerline rise on all three meshes. Keep the mesh-spacing and temperature units visible, and distinguish the observed order, extrapolated rise, and GCI estimate in your report.

**Deliverables:** a reproducible three-mesh table; $p$, $f_{\mathrm{ext}}$, $G$ in kelvin, and $G/|f_{40}|$ as a fraction; plus two sentences describing limitations of the GCI estimate.

**Check:** observed order in $[1.9,2.1]$, extrapolated rise within 0.001 K of 10 K, and compare the estimated $G$ with the exact fine-grid error.

In [ ]:
table = [
    "| Intervals N | Spacing h (m) | Centerline rise f_N (K) | Exact error (K) |",
    "|---:|---:|---:|---:|",
]
for N in MESHES:
    row = mesh_results[N]
    exact_error = abs(row["f"] - AMPLITUDE)
    table.append(
        f"| {N} | {row['h']:.5f} | {row['f']:.8f} | {exact_error:.3e} |"
    )
display(Markdown("\n".join(table)))

# TODO: Add a clearly labeled summary of p, f_ext (K), G (K), G / abs(f40),
# and the exact fine-grid error abs(f40 - AMPLITUDE).

## 5. Solution and diagnostic plots

Plot the three numerical temperature fields together with the analytic solution. Add a second panel for the pointwise error $T_N-T^*$ so mesh differences remain visible even when the temperature curves nearly overlap.

In [ ]:
x_exact = np.linspace(0.0, L, 401)
T_exact = exact_temperature(x_exact)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].plot(x_exact, T_exact, color="black", lw=2, label="Exact")
for N in MESHES:
    row = mesh_results[N]
    axes[0].plot(row["x"], row["T"], "o-", ms=3, label=f"N={N}")
    axes[1].plot(row["x"], row["T"] - exact_temperature(row["x"]),
                 "o-", ms=3, label=f"N={N}")
axes[0].set(xlabel="Position x (m)", ylabel="Temperature T (K)",
            title="Temperature fields")
axes[1].set(xlabel="Position x (m)", ylabel="Numerical - exact (K)",
            title="Nodal temperature error")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.show()

## 6. Automatic checks

Check that each mesh includes the shared center node, boundary values stay fixed, the fields are symmetric, and the centerline rise converges monotonically. After completing the Richardson/GCI calculations, enforce the stated accuracy targets.

In [ ]:
for N in MESHES:
    row = mesh_results[N]
    assert np.isclose(row["x"][N // 2], L / 2)
    assert row["T"][0] == T_LEFT and row["T"][-1] == T_RIGHT
    np.testing.assert_allclose(row["T"], row["T"][::-1], rtol=0, atol=1e-10)

f_values = np.array([mesh_results[N]["f"] for N in MESHES])
center_errors = np.abs(f_values - AMPLITUDE)
assert np.all(np.diff(f_values) < 0), "Expected centerline rise to converge monotonically."
assert np.all(np.diff(center_errors) < 0), "Centerline error should decrease with refinement."

# TODO: After calculating p and f_ext, assert 1.9 <= p <= 2.1 and
# abs(f_ext - AMPLITUDE) < 0.001. Report G and compare it with abs(f40 - AMPLITUDE).
print("PASS: center-node alignment, fixed boundaries, symmetry, and monotonic centerline convergence.")

## 7. Discussion and next steps

### Submit your findings

- Equation sketch: interior stencil, endpoint contributions, source units, and exact outward-flux/generation balance.
- Mesh sampling: explain why $N=10,20,40$ all include the same center location.
- Results: report $f_{10},f_{20},f_{40}$, observed $p$, extrapolated rise $f_{\mathrm{ext}}$, and the fine-grid estimate $G$ in kelvin and as $G/|f_{40}|$.
- Comparison: compare $G$ with the exact error $|f_{40}-10\,\mathrm{K}|$.
- Limitations: write two sentences about assumptions or uncertainties not represented by this numerical estimate. State why GCI is not a guaranteed confidence interval.

### Scope

This smooth manufactured example lets you compare a mesh-error estimate with a known exact answer. A successful check here does not establish that a physical model, its inputs, or every future mesh is accurate.

### Reading

Study the NASA sections on Richardson extrapolation and the grid convergence index, plus Section 4.1 of the assigned NIST reference. Practice time: 40 minutes; reading: 25 minutes.